# Neural CFR+ 18-Claim LR-Reset Forks

This notebook tests whether the late-run deterioration is caused by optimizer / learning-rate dynamics rather than an unavoidable CFR+ averaging effect.

Structure:

1. Train one common neural CFR+ trunk for 60 measured neural-training minutes.
2. Save a resumable exact checkpoint.
3. Fork that exact state into three continuations:
   - baseline continuation,
   - learning-rate drop plus optimizer reset,
   - learning-rate drop plus optimizer reset plus more traversals.

Exact evaluation time is excluded from the measured training budget. Checkpoints are overwritten atomically inside each phase so the notebook can recover from an interruption without creating many large checkpoint files.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
from pathlib import Path
import sys
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


def find_repo_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "liars_poker").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Could not find repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer
from liars_poker.core import GameSpec
from liars_poker.serialization import save_policy
from liars_poker.training.deep_cfr_plus import _exact_evaluation

print("repo:", REPO_ROOT)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    torch.set_float32_matmul_precision("high")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=("RankHigh", "Pair", "TwoPair", "Trips"),
    suit_symmetry=True,
)

BASELINE_MINUTES = 60

# 60 + 3 * 100 = 360 measured neural-training minutes, before exact-eval overhead.
# Change this to 60 for a shorter 4-hour measured-training version.
FORK_MINUTES = 100

EVALUATE_EVERY_MINUTES = 10
CHECKPOINT_EVERY_MINUTES = 30
SEED = 7

BASE_TRAVERSALS_PER_PLAYER = 1024
MORE_TRAVERSALS_PER_PLAYER = 4096

TRAINER_KWARGS = {
    "regret_hidden_sizes": (2048, 2048),
    "strategy_hidden_sizes": (512, 512),
    "device": device,
    "seed": SEED,
    "regret_buffer_capacity": 2_000_000,
    "strategy_buffer_capacity": 2_000_000,
    "learning_rate": 1e-3,
    "batch_size": 4096,
    "regret_train_steps": 24,
    "strategy_train_steps": 6,
    "regret_positive_weight": 0.5,
    "strategy_weighting": "linear",
    "validation_fraction": 0.02,
    "validation_buffer_capacity": 20_000,
    "traversal_backend": "gpu_native",
    "traversal_batch_size": 1024,
    "device_replay": True,
    "fused_optimizer": True,
    "amp_dtype": None,
    "compile_models": False,
}

run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
RUN_DIR = REPO_ROOT / "artifacts" / "cfr_plus_18_claim_lr_reset_forks" / f"{spec.to_short_str()}___{run_id}"
BASE_DIR = RUN_DIR / "base_60m"
FORK_ROOT = RUN_DIR / "forks"
RUN_DIR.mkdir(parents=True, exist_ok=True)

FORKS = [
    {
        "name": "baseline_continue",
        "label": "baseline continuation",
        "learning_rate": 1e-3,
        "reset_optimizers": False,
        "traversals_per_player": BASE_TRAVERSALS_PER_PLAYER,
    },
    {
        "name": "lr_drop_optimizer_reset",
        "label": "LR 1e-4 + optimizer reset",
        "learning_rate": 1e-4,
        "reset_optimizers": True,
        "traversals_per_player": BASE_TRAVERSALS_PER_PLAYER,
    },
    {
        "name": "lr_drop_reset_more_traversals",
        "label": "LR 1e-4 + reset + 4096 traversals",
        "learning_rate": 1e-4,
        "reset_optimizers": True,
        "traversals_per_player": MORE_TRAVERSALS_PER_PLAYER,
    },
]

manifest = {
    "run_type": "cfr_plus_18_claim_lr_reset_forks",
    "spec": json.loads(spec.to_json()),
    "seed": SEED,
    "baseline_minutes": BASELINE_MINUTES,
    "fork_minutes": FORK_MINUTES,
    "evaluate_every_minutes": EVALUATE_EVERY_MINUTES,
    "checkpoint_every_minutes": CHECKPOINT_EVERY_MINUTES,
    "base_traversals_per_player": BASE_TRAVERSALS_PER_PLAYER,
    "trainer_kwargs": {**TRAINER_KWARGS, "device": str(device)},
    "forks": FORKS,
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
print("Run directory:", RUN_DIR)

In [ ]:
def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, (tuple, set)):
        return list(value)
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, default=json_default), encoding="utf-8")
    tmp.replace(path)


def append_jsonl(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, default=json_default) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def atomic_checkpoint(trainer: DeepCFRPlusTrainer, path: Path) -> float:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".new")
    start = time.perf_counter()
    trainer.save_checkpoint(tmp)
    os.replace(tmp, path)
    return time.perf_counter() - start


def checkpoint_size_gib(path: Path) -> float:
    if not path.exists():
        return 0.0
    return path.stat().st_size / (1024 ** 3)


def cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def reset_lr_and_optimizers(trainer: DeepCFRPlusTrainer, learning_rate: float) -> None:
    trainer.learning_rate = float(learning_rate)
    trainer.regret_optimizers = [trainer._make_optimizer(model) for model in trainer.regret_nets]
    trainer.strategy_optimizers = [trainer._make_optimizer(model) for model in trainer.strategy_nets]
    trainer._compiled_forwards.clear()


def run_exact_eval(
    trainer: DeepCFRPlusTrainer,
    *,
    phase_dir: Path,
    phase_name: str,
    phase_label: str,
    measured_phase_s: float,
    measured_total_s: float,
    traversals_per_player: int,
) -> dict:
    start = time.perf_counter()
    metrics = _exact_evaluation(spec, trainer)
    evaluation_s = time.perf_counter() - start
    record = {
        "phase": phase_name,
        "label": phase_label,
        "iteration": trainer.iteration,
        "measured_phase_s": measured_phase_s,
        "measured_phase_min": measured_phase_s / 60.0,
        "measured_total_s": measured_total_s,
        "measured_total_min": measured_total_s / 60.0,
        "traversals_per_player": int(traversals_per_player),
        "evaluation_s": evaluation_s,
        **metrics,
    }
    record["exploitability"] = float(2.0 * record["predicted_avg"] - 1.0)
    record["current_exploitability"] = float(2.0 * record["current_predicted_avg"] - 1.0)
    append_jsonl(phase_dir / "evaluations.jsonl", record)
    print(
        f"[eval] {phase_label}: total={record['measured_total_min']:.1f}m "
        f"iter={trainer.iteration} avg_exp={record['exploitability']:.5f} "
        f"current_exp={record['current_exploitability']:.5f} "
        f"eval={evaluation_s:.1f}s"
    )
    return record


def summarize_iteration_record(
    record: dict,
    *,
    phase_name: str,
    phase_label: str,
    measured_phase_s: float,
    measured_total_s: float,
    traversals_per_player: int,
) -> dict:
    timing = record.get("timing", {})
    action_sampling = record.get("action_sampling", {})
    validation = record.get("validation", {})
    row = {
        "phase": phase_name,
        "label": phase_label,
        "iteration": record.get("iteration"),
        "measured_phase_s": measured_phase_s,
        "measured_phase_min": measured_phase_s / 60.0,
        "measured_total_s": measured_total_s,
        "measured_total_min": measured_total_s / 60.0,
        "traversals_per_player": int(traversals_per_player),
        "traversal_s": timing.get("traversal_s"),
        "regret_training_s": timing.get("regret_training_s"),
        "strategy_training_s": timing.get("strategy_training_s"),
        "new_regret_records": sum(record.get("regret_records_added", []) or []),
        "new_strategy_records": sum(record.get("strategy_records_added", []) or []),
        "regret_records_seen": sum(record.get("regret_records_seen", []) or []),
        "strategy_records_seen": sum(record.get("strategy_records_seen", []) or []),
        "regret_buffer_size": sum(record.get("regret_buffer_sizes", []) or []),
        "strategy_buffer_size": sum(record.get("strategy_buffer_sizes", []) or []),
        "regret_loss": np.nanmean(record.get("regret_losses", [np.nan])),
        "strategy_loss": np.nanmean(record.get("strategy_losses", [np.nan])),
        "claim_edge_fraction": action_sampling.get("claim_edge_fraction"),
        "regret_weight_ess_fraction": action_sampling.get("regret_weight_ess_fraction"),
        "largest_regret_weight": action_sampling.get("largest_regret_weight"),
    }
    for group_name, group in validation.items() if isinstance(validation, dict) else []:
        if isinstance(group, dict):
            for key, value in group.items():
                row[f"validation_{group_name}_{key}"] = value
    return row


def phase_state_path(phase_dir: Path) -> Path:
    return phase_dir / "state.json"


def phase_summary_path(phase_dir: Path) -> Path:
    return phase_dir / "summary.json"

In [ ]:
def run_phase(
    *,
    phase_dir: Path,
    phase_name: str,
    phase_label: str,
    budget_minutes: float,
    traversals_per_player: int,
    source_checkpoint: Path | None = None,
    base_total_s: float = 0.0,
    trainer_kwargs: dict | None = None,
    learning_rate: float | None = None,
    reset_optimizers: bool = False,
) -> dict:
    phase_dir.mkdir(parents=True, exist_ok=True)
    summary_path = phase_summary_path(phase_dir)
    checkpoint_path = phase_dir / "latest_checkpoint.pt"
    state_path = phase_state_path(phase_dir)

    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        if summary.get("status") == "complete":
            print("Skipping completed phase:", phase_label)
            return summary

    measured_phase_s = 0.0
    resumed_own_checkpoint = checkpoint_path.exists() and state_path.exists()
    if resumed_own_checkpoint:
        state = json.loads(state_path.read_text(encoding="utf-8"))
        measured_phase_s = float(state.get("measured_phase_s", 0.0))
        base_total_s = float(state.get("base_total_s", base_total_s))
        print(f"Resuming phase {phase_label} from {measured_phase_s / 60.0:.2f}m")
        trainer = DeepCFRPlusTrainer.load_checkpoint(checkpoint_path, device=device)
    elif source_checkpoint is not None:
        print("Loading source checkpoint:", source_checkpoint)
        trainer = DeepCFRPlusTrainer.load_checkpoint(source_checkpoint, device=device)
        if learning_rate is not None:
            trainer.learning_rate = float(learning_rate)
        if reset_optimizers:
            reset_lr_and_optimizers(trainer, float(learning_rate or trainer.learning_rate))
    else:
        trainer = DeepCFRPlusTrainer(spec, **(trainer_kwargs or {}))

    budget_s = 60.0 * float(budget_minutes)
    eval_period_s = 60.0 * float(EVALUATE_EVERY_MINUTES)
    checkpoint_period_s = 60.0 * float(CHECKPOINT_EVERY_MINUTES)
    next_eval_s = (math.floor(measured_phase_s / eval_period_s) + 1) * eval_period_s
    next_checkpoint_s = (math.floor(measured_phase_s / checkpoint_period_s) + 1) * checkpoint_period_s
    last_print_min = int(measured_phase_s // 60) - 1
    final_eval = None

    write_json(
        state_path,
        {
            "status": "running",
            "phase": phase_name,
            "label": phase_label,
            "measured_phase_s": measured_phase_s,
            "base_total_s": base_total_s,
            "measured_total_s": base_total_s + measured_phase_s,
            "iteration": trainer.iteration,
            "traversals_per_player": int(traversals_per_player),
            "learning_rate": float(trainer.learning_rate),
        },
    )

    while measured_phase_s < budget_s:
        start = time.perf_counter()
        record = trainer.run_iteration(traversals_per_player=int(traversals_per_player))
        iteration_s = time.perf_counter() - start
        measured_phase_s += iteration_s
        measured_total_s = base_total_s + measured_phase_s

        if trainer.validation_fraction > 0.0:
            record["validation"] = trainer.validation_metrics()

        row = summarize_iteration_record(
            record,
            phase_name=phase_name,
            phase_label=phase_label,
            measured_phase_s=measured_phase_s,
            measured_total_s=measured_total_s,
            traversals_per_player=traversals_per_player,
        )
        append_jsonl(phase_dir / "training.jsonl", row)

        current_min = int(measured_phase_s // 60)
        if current_min > last_print_min:
            timing = record.get("timing", {})
            print(
                f"{phase_label}: phase={measured_phase_s / 60.0:.1f}/{budget_minutes:.0f}m "
                f"total={measured_total_s / 60.0:.1f}m iter={trainer.iteration} "
                f"trav={timing.get('traversal_s', float('nan')):.2f}s "
                f"regfit={timing.get('regret_training_s', float('nan')):.2f}s "
                f"strfit={timing.get('strategy_training_s', float('nan')):.2f}s"
            )
            last_print_min = current_min

        if measured_phase_s >= next_eval_s:
            final_eval = run_exact_eval(
                trainer,
                phase_dir=phase_dir,
                phase_name=phase_name,
                phase_label=phase_label,
                measured_phase_s=measured_phase_s,
                measured_total_s=measured_total_s,
                traversals_per_player=traversals_per_player,
            )
            next_eval_s += eval_period_s

        if measured_phase_s >= next_checkpoint_s:
            ckpt_s = atomic_checkpoint(trainer, checkpoint_path)
            print(
                f"[checkpoint] {phase_label}: phase={measured_phase_s / 60.0:.1f}m "
                f"size={checkpoint_size_gib(checkpoint_path):.2f} GiB time={ckpt_s:.1f}s"
            )
            next_checkpoint_s += checkpoint_period_s

        write_json(
            state_path,
            {
                "status": "running",
                "phase": phase_name,
                "label": phase_label,
                "measured_phase_s": measured_phase_s,
                "base_total_s": base_total_s,
                "measured_total_s": measured_total_s,
                "iteration": trainer.iteration,
                "traversals_per_player": int(traversals_per_player),
                "learning_rate": float(trainer.learning_rate),
            },
        )

    measured_total_s = base_total_s + measured_phase_s
    if final_eval is None or final_eval.get("iteration") != trainer.iteration:
        final_eval = run_exact_eval(
            trainer,
            phase_dir=phase_dir,
            phase_name=phase_name,
            phase_label=phase_label,
            measured_phase_s=measured_phase_s,
            measured_total_s=measured_total_s,
            traversals_per_player=traversals_per_player,
        )

    policy_dir = phase_dir / "policy"
    save_policy(trainer.average_policy(), str(policy_dir))
    ckpt_s = atomic_checkpoint(trainer, checkpoint_path)
    summary = {
        "status": "complete",
        "phase": phase_name,
        "label": phase_label,
        "measured_phase_s": measured_phase_s,
        "measured_phase_min": measured_phase_s / 60.0,
        "base_total_s": base_total_s,
        "measured_total_s": measured_total_s,
        "measured_total_min": measured_total_s / 60.0,
        "iteration": trainer.iteration,
        "traversals_per_player": int(traversals_per_player),
        "learning_rate": float(trainer.learning_rate),
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_gib": checkpoint_size_gib(checkpoint_path),
        "checkpoint_s": ckpt_s,
        "policy_dir": str(policy_dir),
        "final_eval": final_eval,
    }
    write_json(summary_path, summary)
    write_json(
        state_path,
        {
            **summary,
            "status": "complete",
        },
    )
    print(
        f"Completed {phase_label}: total={measured_total_s / 60.0:.1f}m "
        f"avg_exp={final_eval['exploitability']:.5f} "
        f"checkpoint={summary['checkpoint_gib']:.2f} GiB"
    )

    del trainer
    cleanup_cuda()
    return summary

## 1. Common 60-Minute Trunk

In [ ]:
base_summary = run_phase(
    phase_dir=BASE_DIR,
    phase_name="base_60m",
    phase_label="common 60m trunk",
    budget_minutes=BASELINE_MINUTES,
    traversals_per_player=BASE_TRAVERSALS_PER_PLAYER,
    trainer_kwargs=TRAINER_KWARGS,
)
base_summary

## 2. Fork A: Baseline Continuation

In [ ]:
fork_a = FORKS[0]
fork_a_summary = run_phase(
    phase_dir=FORK_ROOT / fork_a["name"],
    phase_name=fork_a["name"],
    phase_label=fork_a["label"],
    budget_minutes=FORK_MINUTES,
    traversals_per_player=fork_a["traversals_per_player"],
    source_checkpoint=BASE_DIR / "latest_checkpoint.pt",
    base_total_s=base_summary["measured_total_s"],
    learning_rate=fork_a["learning_rate"],
    reset_optimizers=fork_a["reset_optimizers"],
)
fork_a_summary

## 3. Fork B: LR Drop + Optimizer Reset

In [ ]:
fork_b = FORKS[1]
fork_b_summary = run_phase(
    phase_dir=FORK_ROOT / fork_b["name"],
    phase_name=fork_b["name"],
    phase_label=fork_b["label"],
    budget_minutes=FORK_MINUTES,
    traversals_per_player=fork_b["traversals_per_player"],
    source_checkpoint=BASE_DIR / "latest_checkpoint.pt",
    base_total_s=base_summary["measured_total_s"],
    learning_rate=fork_b["learning_rate"],
    reset_optimizers=fork_b["reset_optimizers"],
)
fork_b_summary

## 4. Fork C: LR Drop + Optimizer Reset + More Traversals

In [ ]:
fork_c = FORKS[2]
fork_c_summary = run_phase(
    phase_dir=FORK_ROOT / fork_c["name"],
    phase_name=fork_c["name"],
    phase_label=fork_c["label"],
    budget_minutes=FORK_MINUTES,
    traversals_per_player=fork_c["traversals_per_player"],
    source_checkpoint=BASE_DIR / "latest_checkpoint.pt",
    base_total_s=base_summary["measured_total_s"],
    learning_rate=fork_c["learning_rate"],
    reset_optimizers=fork_c["reset_optimizers"],
)
fork_c_summary

## 5. Results

In [ ]:
def phase_dirs():
    dirs = [("common 60m trunk", BASE_DIR)]
    for fork in FORKS:
        dirs.append((fork["label"], FORK_ROOT / fork["name"]))
    return dirs


eval_rows = []
train_rows = []
summary_rows = []
for label, phase_dir in phase_dirs():
    for row in read_jsonl(phase_dir / "evaluations.jsonl"):
        eval_rows.append(row)
    for row in read_jsonl(phase_dir / "training.jsonl"):
        train_rows.append(row)
    summary_path = phase_summary_path(phase_dir)
    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        final_eval = summary.get("final_eval", {})
        summary_rows.append(
            {
                "label": label,
                "iteration": summary.get("iteration"),
                "measured total min": summary.get("measured_total_min"),
                "phase min": summary.get("measured_phase_min"),
                "traversals/player": summary.get("traversals_per_player"),
                "learning rate": summary.get("learning_rate"),
                "final average exploitability": final_eval.get("exploitability"),
                "final current exploitability": final_eval.get("current_exploitability"),
                "checkpoint GiB": summary.get("checkpoint_gib"),
                "checkpoint path": summary.get("checkpoint_path"),
            }
        )

eval_df = pd.DataFrame(eval_rows)
train_df = pd.DataFrame(train_rows)
summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
if not eval_df.empty:
    display(
        eval_df.sort_values(["label", "measured_total_min"])[
            [
                "label",
                "measured_phase_min",
                "measured_total_min",
                "iteration",
                "traversals_per_player",
                "exploitability",
                "current_exploitability",
                "evaluation_s",
            ]
        ].tail(30)
    )

if not train_df.empty:
    timing = (
        train_df.groupby("label")
        .agg(
            iterations=("iteration", "count"),
            final_total_min=("measured_total_min", "max"),
            mean_traversal_s=("traversal_s", "mean"),
            mean_regret_fit_s=("regret_training_s", "mean"),
            mean_strategy_fit_s=("strategy_training_s", "mean"),
            final_regret_loss=("regret_loss", "last"),
            final_strategy_loss=("strategy_loss", "last"),
            records_seen=("regret_records_seen", "last"),
            strategy_seen=("strategy_records_seen", "last"),
        )
        .sort_values("final_total_min")
    )
    display(timing)

In [ ]:
def prefix_plus_branch(branch_label: str) -> pd.DataFrame:
    base = eval_df[eval_df["label"] == "common 60m trunk"].copy()
    branch = eval_df[eval_df["label"] == branch_label].copy()
    combined = pd.concat([base, branch], ignore_index=True)
    return combined.sort_values("measured_total_min")


fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharex=True)
if not eval_df.empty:
    for fork in FORKS:
        label = fork["label"]
        curve = prefix_plus_branch(label)
        axes[0].plot(
            curve["measured_total_min"],
            curve["exploitability"],
            marker="o",
            label=label,
        )
        axes[1].plot(
            curve["measured_total_min"],
            curve["current_exploitability"],
            marker="o",
            label=label,
        )

axes[0].set_title("Average-policy exact exploitability")
axes[1].set_title("Current-strategy exact exploitability")
for ax in axes:
    ax.set_xlabel("Measured neural-training minutes")
    ax.set_ylabel("Exploitability")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.show()

In [ ]:
if not train_df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True)
    for label, group in train_df.groupby("label"):
        group = group.sort_values("measured_total_min")
        axes[0, 0].plot(group["measured_total_min"], group["regret_loss"], label=label, alpha=0.8)
        axes[0, 1].plot(group["measured_total_min"], group["strategy_loss"], label=label, alpha=0.8)
        axes[1, 0].plot(group["measured_total_min"], group["regret_training_s"], label=label, alpha=0.6)
        axes[1, 1].plot(group["measured_total_min"], group["strategy_training_s"], label=label, alpha=0.6)

    axes[0, 0].set_title("Regret fitting loss")
    axes[0, 1].set_title("Average-network fitting loss")
    axes[1, 0].set_title("Regret fit seconds")
    axes[1, 1].set_title("Strategy fit seconds")
    for ax in axes.ravel():
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.show()

In [ ]:
# Robust annotated comparison plot.
# Run this after the cell that builds plot_df. It annotates staged schedule phase changes
# from STAGED_SCHEDULE/STAGED_ROOT when available, otherwise falls back to train_df.

COMMON_LABEL = "common 60m trunk"
STAGED_LABEL = "best guess staged schedule"


def _read_jsonl_if_exists(path):
    if not path.exists():
        return []
    return read_jsonl(path)


def _phase_label_from_dict(row):
    pieces = []
    lr = row.get("learning_rate")
    if lr is not None and pd.notna(lr):
        pieces.append(f"lr={float(lr):.0e}")
    trav = row.get("traversals_per_player")
    if trav is not None and pd.notna(trav):
        pieces.append(f"trav={int(trav)}")
    regret_steps = row.get("regret_train_steps")
    if regret_steps is not None and pd.notna(regret_steps):
        pieces.append(f"R={int(regret_steps)}")
    strategy_steps = row.get("strategy_train_steps")
    if strategy_steps is not None and pd.notna(strategy_steps):
        pieces.append(f"S={int(strategy_steps)}")
    return "\n".join(pieces)


def _staged_phase_changes_from_dirs():
    if "STAGED_ROOT" not in globals() or "STAGED_SCHEDULE" not in globals():
        return pd.DataFrame()
    rows = []
    for stage in STAGED_SCHEDULE:
        stage_dir = STAGED_ROOT / stage["name"]
        train_rows = _read_jsonl_if_exists(stage_dir / "training.jsonl")
        eval_rows = _read_jsonl_if_exists(stage_dir / "evaluations.jsonl")
        all_rows = train_rows or eval_rows
        if not all_rows:
            continue
        first = pd.DataFrame(all_rows).sort_values(["measured_total_min", "iteration"]).iloc[0]
        rows.append({
            "stage": stage["name"],
            "measured_total_min": float(first["measured_total_min"]),
            "iteration": int(first["iteration"]),
            "learning_rate": stage.get("learning_rate"),
            "traversals_per_player": stage.get("traversals_per_player"),
            "regret_train_steps": stage.get("regret_train_steps"),
            "strategy_train_steps": stage.get("strategy_train_steps"),
        })
    return pd.DataFrame(rows).drop_duplicates("stage") if rows else pd.DataFrame()


def _staged_phase_changes_from_train_df():
    if "train_df" not in globals() or train_df.empty or "label" not in train_df:
        return pd.DataFrame()
    if STAGED_LABEL not in set(train_df["label"].dropna()):
        return pd.DataFrame()
    rows = train_df[train_df["label"] == STAGED_LABEL].copy().sort_values("measured_total_min")
    cols = [
        c for c in [
            "learning_rate",
            "traversals_per_player",
            "regret_train_steps",
            "strategy_train_steps",
        ]
        if c in rows.columns
    ]
    if not cols:
        return pd.DataFrame()
    changed = rows[cols].ne(rows[cols].shift()).any(axis=1)
    out = rows.loc[changed, ["measured_total_min", "iteration", *cols]].copy()
    out["stage"] = [f"phase_{i}" for i in range(len(out))]
    return out.reset_index(drop=True)


phase_df = _staged_phase_changes_from_dirs()
if phase_df.empty:
    phase_df = _staged_phase_changes_from_train_df()

if "plot_df" not in globals():
    # Fallback for older notebook state: construct a simple plot_df from eval_df.
    rows = []
    for label in eval_df["label"].dropna().unique():
        if label == COMMON_LABEL:
            continue
        curve = eval_df[eval_df["label"].isin([COMMON_LABEL, label])].copy()
        curve["curve_label"] = label
        rows.append(curve)
    plot_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
plot_specs = [
    (axes[0, 0], "measured_total_min", "exploitability", "Average-policy exact exploitability vs time", "Measured neural-training minutes"),
    (axes[0, 1], "iteration", "exploitability", "Average-policy exact exploitability vs iterations", "CFR+ iteration"),
    (axes[1, 0], "measured_total_min", "current_exploitability", "Current-strategy exact exploitability vs time", "Measured neural-training minutes"),
    (axes[1, 1], "iteration", "current_exploitability", "Current-strategy exact exploitability vs iterations", "CFR+ iteration"),
]

for label, group in plot_df.groupby("curve_label"):
    group = group.sort_values("measured_total_min")
    linewidth = 3.0 if label == STAGED_LABEL else 1.8
    alpha = 1.0 if label == STAGED_LABEL else 0.75
    zorder = 5 if label == STAGED_LABEL else 2
    for ax, x_col, y_col, title, xlabel in plot_specs:
        if x_col in group and y_col in group:
            ax.plot(
                group[x_col],
                group[y_col],
                marker="o",
                linewidth=linewidth,
                alpha=alpha,
                zorder=zorder,
                label=label,
            )

if not phase_df.empty:
    for _, row in phase_df.iterrows():
        minute = float(row["measured_total_min"])
        iteration = float(row["iteration"])
        text = _phase_label_from_dict(row)
        for ax in (axes[0, 0], axes[1, 0]):
            ax.axvline(minute, color="black", linestyle="--", linewidth=1, alpha=0.35)
            ymax = ax.get_ylim()[1]
            ax.text(minute, ymax, text, rotation=90, va="top", ha="right", fontsize=8, alpha=0.8)
        for ax in (axes[0, 1], axes[1, 1]):
            ax.axvline(iteration, color="black", linestyle="--", linewidth=1, alpha=0.35)
            ymax = ax.get_ylim()[1]
            ax.text(iteration, ymax, text, rotation=90, va="top", ha="right", fontsize=8, alpha=0.8)
else:
    print("No staged phase-change metadata found. Need STAGED_ROOT/STAGED_SCHEDULE or staged rows in train_df.")

for ax, _, _, title, xlabel in plot_specs:
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Exploitability")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.show()

if not phase_df.empty:
    display(phase_df)

